# D3 · GRASS GIS — hydrology, terrain and environmental modelling

**What GRASS GIS is:** one of the oldest and largest open-source GIS (since 1982), with more than 500 tools.
Its strengths are **raster modelling**: hydrology (where water flows and collects), terrain, visibility, cost surfaces,
and environmental models. Tool names tell you the data type: `r.` raster, `v.` vector, `g.` general, `i.` imagery.

**Daily picture:** QGIS native is a Swiss-army knife; GRASS is a science laboratory with specialised instruments.
You go there when the question is about processes in the landscape: water, slope, visibility, movement.

**Two ways to use it:**
1. **Through QGIS Processing** (what we do): `qrun(grass_alg("r.watershed"), elevation=..., basin=...)`.
   QGIS creates a temporary GRASS workspace, imports your files, runs the tool and exports the result as GeoTIFF/GeoPackage.
2. **GRASS on its own**, with its *project* (formerly *location*: one CRS), *mapset* (your workspace) and *region*
   (the extent and cell size of every calculation). Faster for long workflows; see Level 4.

**Parameter names** are GRASS's own, lower case (`elevation=`, `threshold=`). Flags such as `-s` are passed as `**{"-s": True}`.
Use `qhelp(grass_alg("r.watershed"))` to see them all.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
from geotrain.desktop import qrun, qhelp, qlist, sh, grass_alg, enable_grass, qgis_process_path, quote, minimal_project
OUT = DATA_DIR / "d_out"
OUT.mkdir(exist_ok=True)
V = lambda layer: f"{GPKG}|layername={layer}"       # how QGIS names a layer inside a GeoPackage
DEM = DATA_DIR / "dem.tif"
print("qgis_process:", qgis_process_path())

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import show
enable_grass()
nbh = gpd.read_file(GPKG, layer="neighbourhoods")
houses = gpd.read_file(GPKG, layer="houses")
river = gpd.read_file(GPKG, layer="river")

def sample(tif, gdf):
    """Raster values at the points of a GeoDataFrame (masked where no data)."""
    with rasterio.open(tif) as src:
        return np.ma.array([v[0] for v in src.sample(list(zip(gdf.geometry.x, gdf.geometry.y)), masked=True)])

---
## Level 1 — Basics: statistics, slope and the region

**Goal of this level:** run your first GRASS tools and understand the computational region.

**Daily-life picture:** Before measuring a field, agree on its borders and on the size of the squares you count in.

### Exercise 1.1 — Raster statistics

**Step 1 · Purpose.** What is the **purpose** of `r.univar`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`qrun(grass_alg("r.univar"), map=str(DEM), output=str(OUT / "univar.txt"))`. The output is a text table with `|` as separator.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
qrun(grass_alg("r.____"), map=str(DEM), output=str(OUT / "univar.txt"))
pd.read_csv(OUT / "univar.txt", sep="|").T

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Computes univariate statistics of a raster (count, min, max, mean, standard deviation…).

**Real life:** First look at an elevation model or a temperature map; checking for strange values.

```python
qrun(grass_alg("r.univar"), map=str(DEM), output=str(OUT / "univar.txt"))
pd.read_csv(OUT / "univar.txt", sep="|").T
```

</details>

### Exercise 1.2 — Slope and aspect

**Step 1 · Purpose.** What is the **purpose** of `r.slope.aspect`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`elevation=str(DEM)`, `slope=str(OUT / "g_slope.tif")`, `aspect=str(OUT / "g_aspect.tif")`. By default slope is in degrees.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun(grass_alg("r.slope.aspect"), elevation=str(DEM), ____=str(OUT / "g_slope.tif"), aspect=str(OUT / "g_aspect.tif"))
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
show(rasterio.open(res["slope"]), ax=axs[0], cmap="magma", title="slope (°)")
show(rasterio.open(res["aspect"]), ax=axs[1], cmap="twilight", title="aspect (°)"); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Computes slope (steepness) and aspect (the compass direction a slope faces) from a DEM.

**Real life:** Solar panels work best on south-facing slopes; steep slopes limit building and cycling.

```python
res = qrun(grass_alg("r.slope.aspect"), elevation=str(DEM), slope=str(OUT / "g_slope.tif"), aspect=str(OUT / "g_aspect.tif"))
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
show(rasterio.open(res["slope"]), ax=axs[0], cmap="magma", title="slope (°)")
show(rasterio.open(res["aspect"]), ax=axs[1], cmap="twilight", title="aspect (°)"); plt.show()
```

GRASS aspect is counted **counter-clockwise from east** by default (0 = east, 90 = north). Use flag `-n` for compass degrees from north.

</details>

### Exercise 1.3 — The region: change the cell size on the fly

**Step 1 · Purpose.** What is the **purpose** of `GRASS_REGION_CELLSIZE_PARAMETER`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Run r.slope.aspect again with `GRASS_REGION_CELLSIZE_PARAMETER=200` and compare the raster shapes (120×120 at 50 m vs 30×30 at 200 m).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res200 = qrun(grass_alg("r.slope.aspect"), elevation=str(DEM), slope=str(OUT / "g_slope200.tif"), GRASS_REGION_CELLSIZE_PARAMETER=____)
print(rasterio.open(res["slope"]).shape, "->", rasterio.open(res200["slope"]).shape)
print("max slope:", rasterio.open(res["slope"]).read(1, masked=True).max().round(2), "->", rasterio.open(res200["slope"]).read(1, masked=True).max().round(2))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Every GRASS calculation happens in a *region* (extent + cell size). Through QGIS you can set the cell size; GRASS resamples the input to it.

**Real life:** A quick test run at 200 m before the slow final run at 10 m.

```python
res200 = qrun(grass_alg("r.slope.aspect"), elevation=str(DEM), slope=str(OUT / "g_slope200.tif"), GRASS_REGION_CELLSIZE_PARAMETER=200)
print(rasterio.open(res["slope"]).shape, "->", rasterio.open(res200["slope"]).shape)
print("max slope:", rasterio.open(res["slope"]).read(1, masked=True).max().round(2), "->", rasterio.open(res200["slope"]).read(1, masked=True).max().round(2))
```

Coarser cells smooth the terrain, so the steepest slopes disappear. Always report the cell size of a terrain analysis.

</details>

---
## Level 2 — Core tools: where does the water go?

**Goal of this level:** prepare a DEM for hydrology and compute flow accumulation, basins and streams.

**Daily-life picture:** Pour a bucket of water on a sand model of the town: it flows downhill, gathers in valleys, and each valley collects from its own area.

### Exercise 2.1 — Fill depressions

**Step 1 · Purpose.** What is the **purpose** of `r.fill.dir`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`input=str(DEM)`, `output=` (filled DEM), `direction=` and `areas=` (problem areas) are all required outputs. Count how many cells changed.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun(grass_alg("r.fill.dir"), input=str(DEM), output=str(OUT / "g_filled.tif"),
           direction=str(OUT / "g_dir.tif"), areas=str(OUT / "g_areas.tif"))
before = rasterio.open(DEM).read(1, masked=True)
after = rasterio.open(res["____"]).read(1, masked=True)
print("cells raised:", int((after > before + 1e-4).sum()), "| max raise:", float((after - before).max()).__round__(2), "m")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Removes small pits (cells lower than all neighbours) so water can flow out; also returns flow directions.

**Real life:** Real DEMs have many fake pits (measurement noise); without filling, simulated water gets stuck in them.

```python
res = qrun(grass_alg("r.fill.dir"), input=str(DEM), output=str(OUT / "g_filled.tif"),
           direction=str(OUT / "g_dir.tif"), areas=str(OUT / "g_areas.tif"))
before = rasterio.open(DEM).read(1, masked=True)
after = rasterio.open(res["output"]).read(1, masked=True)
print("cells raised:", int((after > before + 1e-4).sum()), "| max raise:", float((after - before).max()).__round__(2), "m")
```

</details>

### Exercise 2.2 — Flow accumulation

**Step 1 · Purpose.** What is the **purpose** of `r.watershed accumulation=`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`elevation=str(DEM)`, `threshold=400`, `accumulation=`. Plot the log of the absolute value (numbers grow very fast downstream).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
ws = qrun(grass_alg("r.watershed"), elevation=str(DEM), threshold=400, **{"-s": True},
          ____=str(OUT / "g_acc.tif"))
acc = rasterio.open(ws["accumulation"]).read(1, masked=True)
ax = plt.subplots(figsize=(5, 5))[1]
show(np.log10(np.abs(acc) + 1), transform=rasterio.open(ws["accumulation"]).transform, ax=ax, cmap="Blues")
river.plot(ax=ax, color="red", linewidth=1); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** For each cell, counts how many upstream cells drain through it. High values = valleys and streams.

**Real life:** Where does rainwater concentrate in a storm? Where should drains or retention ponds go?

```python
ws = qrun(grass_alg("r.watershed"), elevation=str(DEM), threshold=400, **{"-s": True},
          accumulation=str(OUT / "g_acc.tif"))
acc = rasterio.open(ws["accumulation"]).read(1, masked=True)
ax = plt.subplots(figsize=(5, 5))[1]
show(np.log10(np.abs(acc) + 1), transform=rasterio.open(ws["accumulation"]).transform, ax=ax, cmap="Blues")
river.plot(ax=ax, color="red", linewidth=1); plt.show()
```

Negative accumulation means water may flow in from outside the map (the value is an under-estimate). `-s` = single flow direction (D8), easier to interpret.

</details>

### Exercise 2.3 — Watershed basins and streams

**Step 1 · Purpose.** What is the **purpose** of `r.watershed basin= stream= threshold=`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Add `basin=` and `stream=`. Threshold 400 cells × 2 500 m² = 1 km². Count basins with `np.unique`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
ws = qrun(grass_alg("r.watershed"), elevation=str(DEM), threshold=____, **{"-s": True},
          basin=str(OUT / "g_basin.tif"), stream=str(OUT / "g_stream.tif"))
basin = rasterio.open(ws["basin"]).read(1, masked=True)
print(len(np.unique(basin.compressed())), "basins")
ax = plt.subplots(figsize=(5, 5))[1]
show(basin, transform=rasterio.open(ws["basin"]).transform, ax=ax, cmap="tab20")
nbh.boundary.plot(ax=ax, color="white", linewidth=.5); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Splits the land into drainage basins (each drains to one stream segment); `threshold` = minimum basin size in cells.

**Real life:** Which area drains into which stream — the basic unit of water management and flood planning.

```python
ws = qrun(grass_alg("r.watershed"), elevation=str(DEM), threshold=400, **{"-s": True},
          basin=str(OUT / "g_basin.tif"), stream=str(OUT / "g_stream.tif"))
basin = rasterio.open(ws["basin"]).read(1, masked=True)
print(len(np.unique(basin.compressed())), "basins")
ax = plt.subplots(figsize=(5, 5))[1]
show(basin, transform=rasterio.open(ws["basin"]).transform, ax=ax, cmap="tab20")
nbh.boundary.plot(ax=ax, color="white", linewidth=.5); plt.show()
```

</details>

### Exercise 2.4 — Moving-window statistics

**Step 1 · Purpose.** What is the **purpose** of `r.neighbors method=average size=5`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`input=str(DEM)`, `method=0` (average; see `qhelp` for the list), `size=5` (5 × 5 cells), `output=`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
nbr = qrun(grass_alg("r.neighbors"), input=str(DEM), method=0, size=____, output=str(OUT / "g_smooth.tif"))
a, b = rasterio.open(DEM).read(1, masked=True), rasterio.open(nbr["output"]).read(1, masked=True)
print("std before:", round(float(a.std()), 2), "after:", round(float(b.std()), 2))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Replaces each cell by a statistic (average, max…) of the cells around it — a moving window (focal statistics).

**Real life:** Smoothing noisy data; 'average greenness within 125 m of each cell'; local relief = max − min.

```python
nbr = qrun(grass_alg("r.neighbors"), input=str(DEM), method=0, size=5, output=str(OUT / "g_smooth.tif"))
a, b = rasterio.open(DEM).read(1, masked=True), rasterio.open(nbr["output"]).read(1, masked=True)
print("std before:", round(float(a.std()), 2), "after:", round(float(b.std()), 2))
```

</details>

---
## Level 3 — Combining: streams, floods, visibility, movement cost

**Goal of this level:** use GRASS's modelling tools and bring the results back into GeoPandas.

**Daily-life picture:** From 'where is it low?' to 'where would water actually go, and how hard is it to walk there?'

### Exercise 3.1 — Extract a stream network as vectors

**Step 1 · Purpose.** What is the **purpose** of `r.stream.extract stream_vector=`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`elevation=str(DEM)`, `threshold=800`, `stream_vector=str(OUT / "g_streams.gpkg")`, `GRASS_OUTPUT_TYPE_PARAMETER=2` (lines). Then compare with the real river: mean distance of stream vertices to the river.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
st = qrun(grass_alg("r.stream.extract"), elevation=str(DEM), threshold=____, stream_vector=str(OUT / "g_streams.gpkg"),
          GRASS_OUTPUT_TYPE_PARAMETER=2)
streams = gpd.read_file(st["stream_vector"])
print(len(streams), "stream segments,", round(streams.length.sum() / 1000, 1), "km")
ax = nbh.boundary.plot(color="grey", figsize=(5, 5)); streams.plot(ax=ax, color="blue"); river.plot(ax=ax, color="red", linewidth=2, alpha=.5); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Derives stream lines from the DEM wherever flow accumulation exceeds a threshold, and exports them as vector lines.

**Real life:** Mapping small streams that are missing from official maps; checking that a DEM 'sees' the real river.

```python
st = qrun(grass_alg("r.stream.extract"), elevation=str(DEM), threshold=800, stream_vector=str(OUT / "g_streams.gpkg"),
          GRASS_OUTPUT_TYPE_PARAMETER=2)
streams = gpd.read_file(st["stream_vector"])
print(len(streams), "stream segments,", round(streams.length.sum() / 1000, 1), "km")
ax = nbh.boundary.plot(color="grey", figsize=(5, 5)); streams.plot(ax=ax, color="blue"); river.plot(ax=ax, color="red", linewidth=2, alpha=.5); plt.show()
```

</details>

### Exercise 3.2 — Flood from the river, only where water can reach

**Step 1 · Purpose.** What is the **purpose** of `r.lake`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Seed = the **lowest** point of the river inside the town (sample the DEM along the river). `water_level` = seed elevation + 3.
          `coordinates=f"{x},{y}"`. If the seed is on higher ground than the water level, GRASS stops with an error — the helper shows it.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
line = river.geometry.iloc[0]
pts = gpd.GeoDataFrame(geometry=[line.interpolate(d) for d in np.arange(300, line.length - 300, 50)], crs=river.crs)
elev = sample(DEM, pts)
seed = pts.geometry.iloc[int(elev.argmin())]
lake = qrun(grass_alg("r.lake"), elevation=str(DEM), water_level=float(elev.min()) + ____,
            coordinates=f"{seed.x},{seed.y}", lake=str(OUT / "g_lake.tif"))
depth = sample(lake["lake"], houses)
print(f"seed at {elev.min():.1f} m; flooded houses: {depth.count()}, residents: {houses.residents[~depth.mask].sum()}")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Fills a 'lake' from a seed point up to a water level, flooding only cells **connected** to the seed (unlike the bathtub model in A5).

**Real life:** Which land floods if the river rises 3 m at the town outlet?

```python
line = river.geometry.iloc[0]
pts = gpd.GeoDataFrame(geometry=[line.interpolate(d) for d in np.arange(300, line.length - 300, 50)], crs=river.crs)
elev = sample(DEM, pts)
seed = pts.geometry.iloc[int(elev.argmin())]
lake = qrun(grass_alg("r.lake"), elevation=str(DEM), water_level=float(elev.min()) + 3,
            coordinates=f"{seed.x},{seed.y}", lake=str(OUT / "g_lake.tif"))
depth = sample(lake["lake"], houses)
print(f"seed at {elev.min():.1f} m; flooded houses: {depth.count()}, residents: {houses.residents[~depth.mask].sum()}")
```

</details>

### Exercise 3.3 — Visibility from the lookout tower

**Step 1 · Purpose.** What is the **purpose** of `r.viewshed`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`input=str(DEM)`, `coordinates="391225,5822975"`, `observer_elevation=30`, flag `-b` (1 = visible, 0 = not).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
vs = qrun(grass_alg("r.viewshed"), input=str(DEM), coordinates="391225,5822975", observer_elevation=____,
          **{"-b": True}, output=str(OUT / "g_view.tif"))
print("visible share:", round(float(rasterio.open(vs["output"]).read(1, masked=True).mean()), 3))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Computes which cells are visible from an observer point (with height above ground), considering the terrain.

**Real life:** Visual impact of a wind turbine or a tower; placing CCTV or fire lookouts.

```python
vs = qrun(grass_alg("r.viewshed"), input=str(DEM), coordinates="391225,5822975", observer_elevation=30,
          **{"-b": True}, output=str(OUT / "g_view.tif"))
print("visible share:", round(float(rasterio.open(vs["output"]).read(1, masked=True).mean()), 3))
```

Compare with GDAL `gdal_viewshed` (D2 4.2): two engines, same idea — they should roughly agree.

</details>

### Exercise 3.4 — Cost distance: effort to reach a clinic

**Step 1 · Purpose.** What is the **purpose** of `r.cost`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Make a friction raster with Rasterio: `1 + slope / 2` (flat = 1, steeper = harder). Then `input=friction`, `start_points=V("clinics")`, `output=`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with rasterio.open(OUT / "g_slope.tif") as s:
    prof = s.profile; fr = 1 + s.read(1, masked=True).filled(0) / ____
with rasterio.open(OUT / "friction.tif", "w", **prof) as d:
    d.write(fr.astype("float32"), 1)
cost = qrun(grass_alg("r.cost"), input=str(OUT / "friction.tif"), start_points=V("clinics"), output=str(OUT / "g_cost.tif"))
houses["effort"] = sample(cost["output"], houses).filled(np.nan)
ax = plt.subplots(figsize=(5, 5))[1]
show(rasterio.open(cost["output"]), ax=ax, cmap="viridis"); gpd.read_file(GPKG, layer="clinics").plot(ax=ax, color="red"); plt.show()
houses["effort"].describe().round(1)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Accumulates the 'cost' of moving from start points across a friction raster (each cell has a cost to cross). The result is an effort distance, not metres.

**Real life:** Walking effort to the nearest clinic when slopes make walking harder; wildlife movement across a landscape.

```python
with rasterio.open(OUT / "g_slope.tif") as s:
    prof = s.profile; fr = 1 + s.read(1, masked=True).filled(0) / 2
with rasterio.open(OUT / "friction.tif", "w", **prof) as d:
    d.write(fr.astype("float32"), 1)
cost = qrun(grass_alg("r.cost"), input=str(OUT / "friction.tif"), start_points=V("clinics"), output=str(OUT / "g_cost.tif"))
houses["effort"] = sample(cost["output"], houses).filled(np.nan)
ax = plt.subplots(figsize=(5, 5))[1]
show(rasterio.open(cost["output"]), ax=ax, cmap="viridis"); gpd.read_file(GPKG, layer="clinics").plot(ax=ax, color="red"); plt.show()
houses["effort"].describe().round(1)
```

`r.walk` is the more realistic version: it uses a walking-time formula for uphill and downhill movement.

</details>

---
## Level 4 — Professional: environmental modelling questions

**Goal of this level:** use GRASS results to answer modelling and decision questions, and run GRASS on its own.

**Daily-life picture:** An environmental consultant: 'which basin needs flood protection first, and why?'

### Exercise 4.1 — Which drainage basin holds most people?

**Scenario.** Using the basins from 2.3: how many residents live in each basin? Which basin should get the first storm-water plan?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Sample the basin raster at every house → groupby basin → sum residents. Houses in cells with no basin get -1: keep them apart. Map the top basin.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
houses["basin"] = sample(OUT / "g_basin.tif", houses).filled(-1).astype(int)
print("residents without a basin (water drains off the map edge):", houses.loc[houses.basin < 0, "residents"].sum())
per_basin = houses[houses.basin >= 0].groupby("basin")["residents"].sum().sort_values(ascending=False)
print(per_basin.head())
top = per_basin.index[____]
ax = plt.subplots(figsize=(5, 5))[1]
show((basin == top).astype("uint8"), transform=rasterio.open(OUT / "g_basin.tif").transform, ax=ax, cmap="Reds")
nbh.boundary.plot(ax=ax, color="grey"); plt.show()

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Modelling (hydrological units + population)

Basins, not neighbourhoods, are the right unit for water questions: rain does not care about administrative borders. Cells near the map edge have no complete basin — in a real study, use a DEM that covers the whole catchment, not only the town. Combining a physical unit (basin) with a social variable (residents) is typical environmental modelling.

```python
houses["basin"] = sample(OUT / "g_basin.tif", houses).filled(-1).astype(int)
print("residents without a basin (water drains off the map edge):", houses.loc[houses.basin < 0, "residents"].sum())
per_basin = houses[houses.basin >= 0].groupby("basin")["residents"].sum().sort_values(ascending=False)
print(per_basin.head())
top = per_basin.index[0]
ax = plt.subplots(figsize=(5, 5))[1]
show((basin == top).astype("uint8"), transform=rasterio.open(OUT / "g_basin.tif").transform, ax=ax, cmap="Reds")
nbh.boundary.plot(ax=ax, color="grey"); plt.show()
```

</details>

### Exercise 4.2 — Bathtub vs connected flood: how different are the models?

**Scenario.** Compare three flood models for a 3 m rise: (A) **bathtub** — every cell below *median river level + 3 m* floods (A5 idea);
(B) **r.lake** from the lowest river point + 3 m (3.2); (C) r.lake from the **median** river point + 3 m.
Count flooded houses in each and explain the differences.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Bathtub: `sample(DEM, houses) < np.ma.median(elev) + 3`. For (C), seed = the river point whose elevation is closest to the median.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
lvl = float(np.ma.median(elev))
a = int((sample(DEM, houses) < lvl + 3).sum())
b = int(depth.count())
seed_c = pts.geometry.iloc[int(np.abs(elev - lvl).argmin())]
lake_c = qrun(grass_alg("r.lake"), elevation=str(DEM), water_level=lvl + ____, coordinates=f"{seed_c.x},{seed_c.y}", lake=str(OUT / "g_lake_c.tif"))
c = int(sample(lake_c["lake"], houses).count())
pd.Series({"A bathtub": a, "B r.lake lowest point": b, "C r.lake median point": c}, name="flooded houses")

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Modelling (model comparison)

The bathtub model floods every low cell, even hollows the water cannot reach. r.lake floods only connected cells. And the answer depends strongly on **where** the water level is measured, because the river bed is not flat.
**Recommendation:** use connected models (r.lake, or better a hydraulic model / official flood map), and state the reference point and level. Present the range of results, not one number.

```python
lvl = float(np.ma.median(elev))
a = int((sample(DEM, houses) < lvl + 3).sum())
b = int(depth.count())
seed_c = pts.geometry.iloc[int(np.abs(elev - lvl).argmin())]
lake_c = qrun(grass_alg("r.lake"), elevation=str(DEM), water_level=lvl + 3, coordinates=f"{seed_c.x},{seed_c.y}", lake=str(OUT / "g_lake_c.tif"))
c = int(sample(lake_c["lake"], houses).count())
pd.Series({"A bathtub": a, "B r.lake lowest point": b, "C r.lake median point": c}, name="flooded houses")
```

</details>

### Exercise 4.3 — GRASS on its own: a scripted session

**Scenario.** Run a short GRASS workflow **without QGIS**: import the DEM, set the region, compute basins, and report the number of basins. Use a temporary GRASS project.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Write the GRASS commands in a small shell script and run `grass --tmp-location EPSG:32633 --exec bash script.sh` (GRASS 8.4+ also accepts `--tmp-project`). Commands: `r.in.gdal`, `g.region raster=`, `r.watershed`, `r.stats -c`.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
script = OUT / "grass_job.sh"
script.write_text(f"""
r.in.gdal -o input={DEM} output=dem --quiet
g.region raster=____
r.watershed -s elevation=dem threshold=400 basin=basins --quiet
echo "basins: $(r.stats -n basins | wc -l)"
""")
sh(f"grass --tmp-location EPSG:32633 --exec bash {quote(script)}")

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Reproducibility (native GRASS workflow)

A native GRASS session keeps data inside GRASS between steps, so long chains are faster than importing and exporting at every QGIS call. For Python inside GRASS, use `grass.script` (`gs.run_command(...)`) or the `grass.jupyter` package.

```python
script = OUT / "grass_job.sh"
script.write_text(f"""
r.in.gdal -o input={DEM} output=dem --quiet
g.region raster=dem
r.watershed -s elevation=dem threshold=400 basin=basins --quiet
echo "basins: $(r.stats -n basins | wc -l)"
""")
sh(f"grass --tmp-location EPSG:32633 --exec bash {quote(script)}")
```

</details>

---
## 🏁 Final test

Use GRASS tools. For each: **question type → GRASS tools in order → code → interpretation + main assumption.**

### Test 1 — 🛠️ Task question

**A.** Compute the **topographic wetness index** (TWI) with `r.watershed` (`tci=` output) and report the mean TWI at houses in each neighbourhood.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
tw = qrun(grass_alg("r.watershed"), elevation=str(DEM), threshold=400, tci=str(OUT / "g_tci.tif"))
houses["tci"] = sample(tw["tci"], houses).filled(np.nan)
print(houses.merge(nbh[["nb_id", "name"]], on="nb_id").groupby("name")["tci"].mean().round(2).sort_values())
```
High TWI = places where water tends to collect (wet soils, basements at risk).

</details>

### Test 2 — 🛠️ Task question

**B.** Which neighbourhood is **most visible** from the tower (3.3)? Use zonal statistics on the viewshed.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
z = qrun("native:zonalstatisticsfb", INPUT=V("neighbourhoods"), INPUT_RASTER=str(OUT / "g_view.tif"), RASTER_BAND=1,
         COLUMN_PREFIX="vis_", STATISTICS=[2], OUTPUT=str(OUT / "nbh_vis.gpkg"))
print(gpd.read_file(z["OUTPUT"])[["name", "vis_mean"]].sort_values("vis_mean").round(2))
```
GRASS computes, QGIS native summarises — tools from different engines combine freely through files.

</details>

### Test 3 — 🧠 Modelling question

**C · Modelling storm water.** Riverton plans a **retention pond** to protect Oldtown from heavy rain. How would you use GRASS to choose its location?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

1. Fill the DEM (`r.fill.dir`) → flow accumulation and basins (`r.watershed`).
2. Find the basin(s) draining into Oldtown (houses sampled with the basin raster, 4.1).
3. Candidate cells: high accumulation (water arrives there), **upstream** of Oldtown, low slope (`r.slope.aspect`), not built up, public land.
4. Check storage: `r.lake` at the candidate with a planned dam height → pond area and volume.
Recommendation: rank candidates by 'upstream area captured ÷ cost', and test with a rainfall scenario.

</details>

### Test 4 — 🧠 Modelling question

**D · Modelling movement.** You want to model how far elderly residents can walk to a clinic in 10 minutes, with hills. Which GRASS tool and inputs?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

`r.walk`: inputs = DEM (uphill/downhill effort), friction raster (roads easy, fields harder, river impassable except at bridges), start points = clinics.
Output = travel time in seconds → threshold 600 s → join to houses → share per neighbourhood. Lower the walking speed for elderly people (e.g. 3 km/h instead of 5) and compare.

</details>

---
## 🏗️ Project — Storm-water risk per building with GRASS

**The brief.** After a cloudburst, water runs over the ground to low points before it reaches the river. Rate every building's **surface-water
        risk** with GRASS: flow accumulation at the building, drainage basins as polygons, and the connected-flood model (`r.lake`, +3 m).
        Combine them into a risk class and publish it.

**Your data** (made for this project, in `data/projects/`):

| Layer / file | What it holds |
|---|---|
| `dem.tif` | elevation |
| `buildings.gpkg` | footprints with residents |
| `riverton.gpkg → river` | the river |

**How to work:** for each task write the plan in words, then the code, then one sentence with the result. Model answers are hidden; open them only after you tried. The last part happens in **QGIS**.

In [ ]:
from geotrain.projects import build_projects, PROJ_DIR, publish
build_projects()
P = lambda name, layer=None: f"{PROJ_DIR / name}" + (f"|layername={layer}" if layer else "")

bld = gpd.read_file(PROJ_DIR / "buildings.gpkg")

### Project task 1

**Flow accumulation at each building.** `r.watershed` on the DEM, sample accumulation at building centroids (use absolute values). Which buildings are in the top 5 %?

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
ws = qrun(grass_alg("r.watershed"), elevation=str(DEM), threshold=400, **{"-s": True},
          accumulation=str(OUT / "p_acc.tif"), basin=str(OUT / "p_basin.tif"))
cent = bld.centroid
with rasterio.open(ws["accumulation"]) as a:
    bld["acc"] = np.abs(np.array([v[0] for v in a.sample(list(zip(cent.x, cent.y)))], dtype=float))
top = bld["acc"] >= bld["acc"].quantile(0.95)
print(int(top.sum()), "buildings in the top 5 % of flow accumulation,", int(bld.loc[top, "residents"].sum()), "residents")
```

</details>

### Project task 2

**Basins as polygons.** Convert the basin raster to polygons (`r.to.vect`, type area) and count residents per basin.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
bv = qrun(grass_alg("r.to.vect"), input=ws["basin"], type=2, output=str(OUT / "p_basins.gpkg"), GRASS_OUTPUT_TYPE_PARAMETER=3)
basins = gpd.read_file(bv["output"]).dissolve("value").reset_index()
j = gpd.sjoin(bld.assign(geometry=cent), basins[["value", "geometry"]], predicate="within")
basins["residents"] = basins["value"].map(j.groupby("value")["residents"].sum()).fillna(0)
basins.nlargest(3, "residents")[["value", "residents"]]
```

</details>

### Project task 3

**Connected flood (+3 m).** `r.lake` from the lowest river point; flood depth at each building.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
line = river.geometry.iloc[0]
pts = [line.interpolate(d) for d in np.arange(300, line.length - 300, 50)]
with rasterio.open(DEM) as d:
    el = np.ma.array([v[0] for v in d.sample([(p.x, p.y) for p in pts], masked=True)])
seed = pts[int(el.argmin())]
lk = qrun(grass_alg("r.lake"), elevation=str(DEM), water_level=float(el.min()) + 3, coordinates=f"{seed.x},{seed.y}", lake=str(OUT / "p_lake.tif"))
with rasterio.open(lk["lake"]) as l:
    depth = np.ma.array([v[0] for v in l.sample(list(zip(cent.x, cent.y)), masked=True)])
bld["flood_depth"] = depth.filled(0)
print(int((bld.flood_depth > 0).sum()), "buildings flooded")
```

</details>

### Project task 4

**Risk class** per building: 'high' if flooded, 'medium' if in the top 5 % of accumulation, else 'low'. Residents per class?

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
bld["risk"] = np.where(bld.flood_depth > 0, "high", np.where(top, "medium", "low"))
bld.groupby("risk")["residents"].agg(["count", "sum"])
```

</details>

### Project task 5

**Publish** buildings with risk, and basins with residents, to PostGIS.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
publish({"storm_buildings": bld[["bldg_id", "residents", "acc", "flood_depth", "risk", "geometry"]], "storm_basins": basins})
```

</details>

### 🗺️ Project in QGIS

1. Load `storm_buildings` (*Categorized* on `risk`: high red, medium orange, low grey) and `storm_basins` (outline only, labels = `residents`).
2. Add `data/d_out/p_acc.tif` with a *Singleband pseudocolor* on a log scale (or clip values at 500) to see the flow paths.
3. In QGIS: *Processing → GRASS → r.watershed* on `dem.tif` with the same threshold — confirm you get the same basins through the dialog.

<details><summary>🔌 <b>How to connect QGIS to PostGIS</b> (once per computer)</summary>

1. Start the database (Docker: `docker start geotrain-db`).
2. In QGIS, open the **Browser** panel → right-click **PostgreSQL** → **New Connection…**
3. Name `geotrain`, Host `localhost`, Port `5432`, Database `geotrain`.
   Authentication → *Basic*: user `geo`, password `geo` (tick *Store*). Click **Test Connection** → OK.
4. Expand **geotrain → projects** (the schema) and **drag a table onto the map**.
5. To run SQL and see the result on the map: **Database → DB Manager → PostGIS → geotrain → SQL Window**.
   Write the query → **Execute** → tick **Load as new layer**, choose the geometry column and a unique id column → **Load**.
6. To open a GeoPackage instead: drag the `.gpkg` file from your file manager into QGIS.

</details>

### 📦 Deliverables (tick when done)

- [ ] Buildings with accumulation, flood depth and risk class
- [ ] Basins with residents
- [ ] Risk map in QGIS

### 🪞 Reflection — your long-term plan

Is there a water, terrain or visibility question in your research area? Write it, and list the GRASS tools you would chain.

✍️ **Your answer:** …